## Aprendizado por Reforço com SARSA

Neste Jupyter Notebook, implementaremos o algoritmo **SARSA** (*State-Action-Reward-State-Action*), um método de aprendizado por reforço **on-policy**.

### Cenário: O Agente no Corredor

Imagine um agente robô que percorre um corredor linear com **4 posições (estados)**. Em cada posição, ele pode:
- **Ação 0:** Avançar para a próxima posição
- **Ação 1:** Recuar para a posição anterior

O objetivo é chegar à **posição 3 (meta final)**, onde recebe uma recompensa de **+10**. Todos os outros movimentos têm recompensa de **-1** (custo de deslocamento).

---

### SARSA vs Q-Learning: qual é a diferença?

| Critério | Q-Learning | SARSA |
|---|---|---|
| Tipo de política | Off-policy | On-policy |
| Atualização usa | `max Q(s', a')` | `Q(s', a')` da ação **realmente tomada** |
| Comportamento | Mais otimista | Mais conservador |
| Fórmula | `Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]` | `Q(s,a) ← Q(s,a) + α[r + γ·Q(s',a') - Q(s,a)]` |

> **Intuição chave:** No Q-Learning, o agente aprende *como se fosse sempre agir de forma ótima*. No SARSA, ele aprende levando em conta *o que ele realmente faz*, incluindo momentos de exploração aleatória.

## 1. Importação de Bibliotecas e Inicialização

Definimos o ambiente com **4 estados** e **2 ações**.

A Q-table é inicializada com **zeros** (diferente de valores aleatórios), o que é uma prática comum em SARSA para garantir que o agente inicie sem viés de exploração.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definição do ambiente
num_states = 4   # Posições no corredor: 0, 1, 2, 3
num_actions = 2  # Ação 0: avançar | Ação 1: recuar

# Inicialização da Q-table com zeros
# Linha = estado, Coluna = ação
q_table = np.zeros((num_states, num_actions))

print("Q-Table inicial (zeros):")
print(q_table)

## 2. Definição de Hiperparâmetros

Os hiperparâmetros controlam **como** e **quão rápido** o agente aprende:

- **`learning_rate (α)`**: O quanto cada nova experiência modifica o conhecimento atual. Valores altos aprendem rápido, mas podem ser instáveis.
- **`discount_factor (γ)`**: O quanto recompensas futuras importam. `γ = 0.9` significa que o agente valoriza o longo prazo.
- **`epsilon (ε)`**: Probabilidade de tomar uma ação aleatória (exploração). Reduzido ao longo do tempo (*epsilon decay*) para o agente explorar mais no início e explorar menos depois.
- **`num_episodes`**: Número de tentativas completas de percorrer o corredor.

In [ ]:
# Hiperparâmetros
learning_rate   = 0.1    # α: taxa de aprendizado
discount_factor = 0.9    # γ: fator de desconto
epsilon         = 1.0    # ε inicial: 100% exploração no começo
epsilon_min     = 0.05   # ε mínimo: sempre mantém 5% de exploração
epsilon_decay   = 0.95   # fator de decaimento de ε por episódio
num_episodes    = 100    # número de episódios de treinamento

print(f"Configuração: α={learning_rate}, γ={discount_factor}, ε_inicial={epsilon}, episódios={num_episodes}")

## 3. Funções Auxiliares do Ambiente

Aqui definimos duas funções que simulam o ambiente:

- **`escolher_acao`**: Política epsilon-greedy — com probabilidade `ε` escolhe aleatoriamente, caso contrário escolhe a melhor ação conhecida.
- **`executar_acao`**: Recebe o estado atual e a ação, retorna o próximo estado e a recompensa.

> **Regras do corredor:**
> - Avançar (ação 0): vai para `state + 1`, limitado ao máximo 3
> - Recuar (ação 1): vai para `state - 1`, limitado ao mínimo 0
> - Chegar ao estado 3: recompensa +10
> - Qualquer outro movimento: recompensa -1

In [ ]:
def escolher_acao(state, q_table, epsilon):
    """Política epsilon-greedy: explora ou explota."""
    if np.random.uniform(0, 1) < epsilon:
        return np.random.randint(0, num_actions)  # Exploração aleatória
    else:
        return np.argmax(q_table[state, :])        # Exploração do melhor Q-value


def executar_acao(state, action):
    """Simula a transição de estado e retorna (next_state, reward)."""
    if action == 0:  # Avançar
        next_state = min(state + 1, num_states - 1)
    else:            # Recuar
        next_state = max(state - 1, 0)
    
    reward = 10 if next_state == 3 else -1
    return next_state, reward


print("Funções do ambiente definidas com sucesso.")

## 4. Loop de Treinamento SARSA

### A Equação de Atualização do SARSA:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \cdot Q(s', a') - Q(s, a) \right]$$

Onde:
- `s` = estado atual, `a` = ação tomada
- `r` = recompensa recebida
- `s'` = próximo estado, `a'` = **próxima ação que SERÁ tomada** (ponto crítico do SARSA)
- `α` = learning rate, `γ` = discount factor

### O que diferencia o loop do SARSA:

No SARSA, a próxima ação `a'` é **escolhida antes da atualização** e **usada na fórmula**. O agente se compromete com a ação e aprende com ela — não com o hipotético "melhor caso".

In [ ]:
recompensas_por_episodio = []  # Para visualização posterior

for episode in range(num_episodes):
    
    # --- Inicialização do episódio ---
    state  = 0                                      # Agente sempre começa na posição 0
    action = escolher_acao(state, q_table, epsilon) # SARSA: escolhe a primeira ação ANTES do loop
    total_reward = 0
    
    while True:
        
        # 1. Executa a ação e observa o resultado
        next_state, reward = executar_acao(state, action)
        total_reward += reward
        
        # 2. Escolhe a PRÓXIMA ação (on-policy: segue a mesma política ε-greedy)
        next_action = escolher_acao(next_state, q_table, epsilon)
        
        # 3. Atualização SARSA usando Q(s', a') — não max Q(s', :)
        q_table[state, action] = (
            q_table[state, action] +
            learning_rate * (
                reward
                + discount_factor * q_table[next_state, next_action]  # <-- diferença do Q-Learning
                - q_table[state, action]
            )
        )
        
        # 4. Avança para o próximo par (estado, ação)
        state  = next_state
        action = next_action
        
        # 5. Condição de término: chegou à meta
        if state == 3:
            break
    
    # Decaimento de epsilon após cada episódio
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    recompensas_por_episodio.append(total_reward)

print("Treinamento concluído.")
print(f"Epsilon final: {epsilon:.4f}")

## 5. Exibição da Q-Table Final

Cada valor representa o **retorno esperado** ao tomar aquela ação naquele estado, seguindo a política aprendida.

A interpretação ideal para o corredor:
- **Estado 0, 1, 2**: Ação 0 (avançar) deve ter Q-value maior → agente aprendeu a ir em frente
- **Estado 3**: Q-values menores, pois o episódio termina aqui

In [ ]:
print("Q-Table Final — SARSA:\n")
print(f"{'Estado':<10} {'Ação 0 (Avançar)':>20} {'Ação 1 (Recuar)':>20} {'Decisão Ótima':>18}")
print("-" * 72)

for state in range(num_states):
    q0 = q_table[state, 0]
    q1 = q_table[state, 1]
    melhor = "Avançar" if q0 >= q1 else "Recuar"
    print(f"{state:<10} {q0:>20.4f} {q1:>20.4f} {melhor:>18}")

## 6. Visualização: Recompensa Acumulada por Episódio

Este gráfico mostra a **evolução do aprendizado**. Esperamos ver:
- Recompensas inicialmente baixas ou negativas (exploração intensa)
- Convergência para recompensas mais altas à medida que o agente aprende a ir direto à meta

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(recompensas_por_episodio, color='steelblue', linewidth=1.5, label='Recompensa por Episódio')

# Média móvel para visualizar a tendência
janela = 10
media_movel = np.convolve(recompensas_por_episodio, np.ones(janela)/janela, mode='valid')
plt.plot(range(janela - 1, num_episodes), media_movel, color='tomato', linewidth=2.5, label=f'Média Móvel ({janela} ep.)')

plt.xlabel('Episódio')
plt.ylabel('Recompensa Total')
plt.title('Curva de Aprendizado — SARSA')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Teste da Política Aprendida (Execução Greedy)

Após o treinamento, rodamos o agente **sem exploração** (epsilon = 0) para verificar se ele aprendeu a rota ótima: `0 → 1 → 2 → 3`.

In [ ]:
print("Teste da política aprendida (sem exploração):\n")

state = 0
trajeto = [state]
max_steps = 20  # segurança para evitar loop infinito

for step in range(max_steps):
    action = np.argmax(q_table[state, :])  # sempre escolhe a melhor ação
    next_state, reward = executar_acao(state, action)
    acao_nome = 'Avançar' if action == 0 else 'Recuar'
    print(f"  Passo {step+1}: Estado {state} → Ação '{acao_nome}' → Estado {next_state} (recompensa: {reward})")
    state = next_state
    trajeto.append(state)
    if state == 3:
        print(f"\nMeta atingida em {step+1} passo(s)!")
        break
else:
    print("\nAgente não atingiu a meta. Pode ser necessário mais treinamento.")

print(f"Trajeto: {' → '.join(map(str, trajeto))}")

## 8. Resumo Conceitual

### O que aprendemos com este exercício:

| Conceito | O que representa neste exemplo |
|---|---|
| **Estado** | Posição do agente no corredor (0 a 3) |
| **Ação** | Avançar (0) ou Recuar (1) |
| **Recompensa** | +10 ao chegar na posição 3, -1 nos demais |
| **Q-value** | Valor esperado de longo prazo ao tomar uma ação em um estado |
| **On-policy** | O agente aprende com as **ações que realmente executa**, incluindo exploração |
| **Epsilon decay** | Reduz exploração gradualmente conforme o agente ganha confiança |

### Quando usar SARSA sobre Q-Learning?

- Ambientes onde o agente **pagará o custo real** das ações de exploração (ex: robótica física, finanças)
- Quando a **segurança** durante o aprendizado importa mais do que otimizar o retorno máximo teórico
- Políticas que precisam ser **consistentes** entre treinamento e execução

---
> **Referência:** Sutton & Barto, *Reinforcement Learning: An Introduction*, Cap. 6.4 — [link](http://incompleteideas.net/book/the-book.html)